# SGG Text RAG — Development Notebook

Use this notebook to interactively develop and test the Text RAG pipeline before running full scripts.

**Steps covered here:**
1. Preview track document text (what gets embedded)
2. Test Ollama embedding on a single track
3. Initialize the `sgg_text_v1` Qdrant collection
4. Embed a small batch and upsert to Qdrant
5. Run a plain-text query and review results

Once satisfied, run the full embed via:
```
python scripts/sgg_text_embed.py init
python scripts/sgg_text_embed.py embed
```

In [9]:
import sys
import os

repo_root = os.path.abspath('..')
scripts_dir = os.path.abspath('../scripts')
for p in [repo_root, scripts_dir]:
    if p not in sys.path:
        sys.path.insert(0, p)

import duckdb
import requests
from dotenv import load_dotenv

load_dotenv('../.env', override=True)

DUCKDB_PATH = '../dbt_sgg/sgg_prod.duckdb'
VIEW = 'fct_audio_vector_v1'
OLLAMA_URL = os.environ.get('OLLAMA_BASE_URL_HOST', 'http://localhost:11434')
EMBED_MODEL = os.environ.get('OLLAMA_EMBED_MODEL', 'nomic-embed-text')
QDRANT_URL = os.environ.get('QDRANT_URL', 'http://localhost:6335')

print('scripts_dir on path:', scripts_dir)
print('OLLAMA_URL:', OLLAMA_URL)
print('EMBED_MODEL:', EMBED_MODEL)
print('QDRANT_URL:', QDRANT_URL)

scripts_dir on path: /Users/mattkennedy/Projects/sgg/scripts
OLLAMA_URL: http://localhost:11434
EMBED_MODEL: nomic-embed-text
QDRANT_URL: http://localhost:6335


## Step 1 — Preview Track Documents
Review what text will be embedded for each track. Adjust `build_doc()` in `scripts/sgg_text_embed.py` if the output doesn't look right.

In [10]:
from sgg_text_embed import build_doc

con = duckdb.connect(DUCKDB_PATH, read_only=True)
rows = con.execute(f'''
    select file_hash, artist, album, title, date, genre, key_key, key_scale,
           bpm, danceability, loudness_integrated
    from {VIEW}
    limit 10
''').fetch_df().to_dict(orient='records')

for row in rows:
    print(build_doc(row))
    print()

Here Today by The Beach Boys, from the album Pet Sounds (1966). Genre: Rock. Musical key: A major. BPM: 122. Feel: moderate danceability, moderate energy.

Menilmontant by Django Reinhardt, from the album Djangology (2002). Genre: Jazz. Musical key: C major. BPM: 114. Feel: high danceability, quiet and gentle.

Regiment by Brian Eno & David Byrne, from the album My Life In The Bush Of Ghosts (1981). Genre: Rock. Musical key: E minor. BPM: 95. Feel: high danceability, moderate energy.

Moon Rocks by Talking Heads, from the album Speaking in Tongues (1983). Genre: Rock. Musical key: E major. BPM: 124. Feel: high danceability, quiet and gentle.

Those To Come by The Shins, from the album Chutes Too Narrow (2003). Genre: Alternative & Punk. Musical key: D major. BPM: 127. Feel: moderate danceability, moderate energy.

Chain Lightning by Steely Dan, from the album Citizen Steely Dan 1972-1980 [Disc 2] (1975). Genre: Rock. Musical key: D major. BPM: 95. Feel: high danceability, moderate ener

## Step 2 — Test Ollama Embedding (Single Track)
Confirm Ollama is reachable and returns a valid embedding vector.

In [11]:
test_doc = build_doc(rows[0])
print('Document:', test_doc)
print()

resp = requests.post(
    f'{OLLAMA_URL}/api/embeddings',
    json={'model': EMBED_MODEL, 'prompt': test_doc},
    timeout=30
)
resp.raise_for_status()
embedding = resp.json()['embedding']

print(f'Embedding dimension: {len(embedding)}')
print(f'First 5 values: {embedding[:5]}')

Document: Here Today by The Beach Boys, from the album Pet Sounds (1966). Genre: Rock. Musical key: A major. BPM: 122. Feel: moderate danceability, moderate energy.

Embedding dimension: 768
First 5 values: [-0.06907881796360016, -0.03749125450849533, -4.219244003295898, 0.2669176459312439, -0.5597989559173584]


## Step 3 — Initialize Qdrant Collection
Creates `sgg_text_v1` with the correct vector dimension. Safe to re-run — skips if already exists.

In [12]:
from qdrant_client import QdrantClient
from qdrant_client.http import models as rest

embed_dim = len(embedding)
client = QdrantClient(url=QDRANT_URL)

try:
    info = client.get_collection('sgg_text_v1')
    print(f'Collection already exists: {info.points_count} points')
except Exception:
    client.create_collection(
        collection_name='sgg_text_v1',
        vectors_config=rest.VectorParams(size=embed_dim, distance=rest.Distance.COSINE),
    )
    print(f'Collection sgg_text_v1 created (dim={embed_dim}, cosine)')

Collection already exists: 9294 points


## Step 4 — Embed a Small Batch (Test)
Embed 20 tracks and upsert to Qdrant. Review results before running the full collection.

In [ ]:
from sgg_text_embed import build_doc, to_point_id

test_rows = con.execute(f'''
    select file_hash, artist, album, title, date, genre, key_key, key_scale,
           bpm, danceability, loudness_integrated
    from {VIEW}
    limit 20
''').fetch_df().to_dict(orient='records')

points = []
for i, row in enumerate(test_rows, start=1):
    fh = row.get('file_hash')
    doc = build_doc(row)
    resp = requests.post(
        f'{OLLAMA_URL}/api/embeddings',
        json={'model': EMBED_MODEL, 'prompt': doc},
        timeout=30
    )
    vector = resp.json()['embedding']
    points.append(rest.PointStruct(
        id=to_point_id(str(fh)),
        vector=vector,
        payload={
            'file_hash': fh,
            'artist': row.get('artist'),
            'album': row.get('album'),
            'title': row.get('title'),
            'genre': row.get('genre'),
            'key_key': row.get('key_key'),
            'key_scale': row.get('key_scale'),
            'doc': doc,
        }
    ))
    print(f'  [{i}/20] {row.get("artist")} — {row.get("title")}')

client.upsert(collection_name='sgg_text_v1', points=points)
print(f'\nUpserted {len(points)} points to sgg_text_v1')

## Step 5 — Run a Plain-Text Query
Search the collection with a natural language question. Review results for relevance.

In [4]:
query_text = 'slow jazzy late night music'

resp = requests.post(
    f'{OLLAMA_URL}/api/embeddings',
    json={'model': EMBED_MODEL, 'prompt': query_text},
    timeout=30
)
query_vector = resp.json()['embedding']

results = client.query_points(
    collection_name='sgg_text_v1',
    query=query_vector,
    limit=5,
    with_payload=True,
    with_vectors=False,
).points

print(f'Query: "{query_text}"\n')
for i, point in enumerate(results, start=1):
    p = point.payload or {}
    print(f'{i}. [{round(point.score, 4)}] {p.get("artist")} — {p.get("title")} ({p.get("album")})')
    print(f'   {p.get("doc")}')
    print()


Query: "slow jazzy late night music"

1. [0.591] Particle — Road's A Breeze (At 3am) (The Sweet Sounds Of Superfly Vol. 1)
   Road's A Breeze (At 3am) by Particle, from the album The Sweet Sounds Of Superfly Vol. 1 (2002). Genre: Jazz. Musical key: B minor. BPM: 136. Feel: high danceability, moderate energy.

2. [0.5894] Johnny Cash — Cocaine Blues (live) (At Folsom Prison/ At San Quentin)
   Cocaine Blues (live) by Johnny Cash, from the album At Folsom Prison/ At San Quentin (1968). Genre: Country. Musical key: C# minor. BPM: 122. Feel: moderate danceability, quiet and gentle.

3. [0.5852] Claude Debussy — Clouds (Greatest Hits)
   Clouds by Claude Debussy, from the album Greatest Hits (unknown year). Genre: Classical. Musical key: E minor. BPM: 91. Feel: low danceability, quiet and gentle.

4. [0.5841] Valses nobles et sentimentales — V. Presque lent (Ravel: The Piano Concertos, Valses Nobles)
   V. Presque lent by Valses nobles et sentimentales, from the album Ravel: The Piano Conce

## Step 5 Continues — Another Plain-Text Query

In [5]:
query_text = 'upbeat energetic music to wake up to'

resp = requests.post(
    f'{OLLAMA_URL}/api/embeddings',
    json={'model': EMBED_MODEL, 'prompt': query_text},
    timeout=30
)
query_vector = resp.json()['embedding']

results = client.query_points(
    collection_name='sgg_text_v1',
    query=query_vector,
    limit=5,
    with_payload=True,
    with_vectors=False,
).points

print(f'Query: "{query_text}"\n')
for i, point in enumerate(results, start=1):
    p = point.payload or {}
    print(f'{i}. [{round(point.score, 4)}] {p.get("artist")} — {p.get("title")} ({p.get("album")})')
    print(f'   {p.get("doc")}')
    print()


Query: "upbeat energetic music to wake up to"

1. [0.6216] The Roots — Push up Ya Lighter (Illadelph Halflife)
   Push up Ya Lighter by The Roots, from the album Illadelph Halflife (unknown year). Genre: Rap/R&B. Musical key: A minor. BPM: 93. Feel: high danceability, moderate energy.

2. [0.5964] The White Stripes — Icky Thump (Icky Thump)
   Icky Thump by The White Stripes, from the album Icky Thump (2007). Genre: Alternative. Musical key: A major. BPM: 90. Feel: high danceability, loud and energetic.

3. [0.5942] The Aliens — Rox (Astronomy For Dogs)
   Rox by The Aliens, from the album Astronomy For Dogs (2007). Genre: Rock. Musical key: B major. BPM: 91. Feel: high danceability, loud and energetic.

4. [0.5933] The Rolling Stones — Claudine (Some Girls [Deluxe Edition])
   Claudine by The Rolling Stones, from the album Some Girls [Deluxe Edition] (2011). Genre: Rock. Musical key: A major. BPM: 105. Feel: high danceability, loud and energetic.

5. [0.5933] David Byrne — Walk On The

## Step 6 — RAG Answer Layer
Take the top-k retrieved tracks from Step 5 and pass them to `gemma3:27b` to generate a natural-language answer with source citations.

**How it works:**
1. Embed the user's query (same as Step 5)
2. Retrieve top-k tracks from `sgg_text_v1`
3. Format retrieved tracks into a context block
4. Send query + context to `gemma3:27b` via Ollama chat API
5. Print the generated answer

This is true RAG — the LLM answer is grounded in your actual collection, not general knowledge.


In [14]:
CHAT_MODEL = os.environ.get('OLLAMA_CHAT_MODEL', 'gemma3:27b')
TOP_K = 8

query_text = 'melancholy and slow, something to listen to on a rainy day'

# Step 1 — embed the query
resp = requests.post(
    f'{OLLAMA_URL}/api/embeddings',
    json={'model': EMBED_MODEL, 'prompt': query_text},
    timeout=30
)
query_vector = resp.json()['embedding']

# Step 2 — retrieve top-k tracks
results = client.query_points(
    collection_name='sgg_text_v1',
    query=query_vector,
    limit=TOP_K,
    with_payload=True,
    with_vectors=False,
).points

# Step 3 — build context block
context_lines = []
for i, point in enumerate(results, start=1):
    p = point.payload or {}
    context_lines.append(
        f"{i}. {p.get('artist')} — {p.get('title')} ({p.get('album')})\n"
        f"   {p.get('doc')}"
    )
context = '\n'.join(context_lines)

# Step 4 — assemble prompt and call gemma3:27b
prompt = f"""You are a music recommendation assistant. A user asked: "{query_text}"

Here are the most relevant tracks from their vinyl collection:

{context}

Based only on the tracks above, write a short, friendly recommendation. Cite specific tracks by artist and title. Do not invent tracks that aren't listed."""

resp = requests.post(
    f'{OLLAMA_URL}/api/chat',
    json={
        'model': CHAT_MODEL,
        'messages': [{'role': 'user', 'content': prompt}],
        'stream': False,
    },
    timeout=120,
)
resp.raise_for_status()
answer = resp.json()['message']['content']

# Step 5 — print results
print(f'Query: "{query_text}"\n')
print('--- Retrieved Tracks ---')
print(context)
print('\n--- Generated Answer ---')
print(answer)


Query: "melancholy and slow, something to listen to on a rainy day"

--- Retrieved Tracks ---
1. Valses nobles et sentimentales — V. Presque lent (Ravel: The Piano Concertos, Valses Nobles)
   V. Presque lent by Valses nobles et sentimentales, from the album Ravel: The Piano Concertos, Valses Nobles (1994). Genre: Classical. Musical key: C# minor. BPM: 152. Feel: low danceability, quiet and gentle.
2. Valses nobles et sentimentales — II. Assez lent (Ravel: The Piano Concertos, Valses Nobles)
   II. Assez lent by Valses nobles et sentimentales, from the album Ravel: The Piano Concertos, Valses Nobles (1994). Genre: Classical. Musical key: Bb major. BPM: 83. Feel: low danceability, quiet and gentle.
3. Valses nobles et sentimentales — III. Modéré (Ravel: The Piano Concertos, Valses Nobles)
   III. Modéré by Valses nobles et sentimentales, from the album Ravel: The Piano Concertos, Valses Nobles (1994). Genre: Classical. Musical key: G major. BPM: 140. Feel: low danceability, quiet and ge

## Cleanup — Delete Test Points Before Full Embed

Run this cell to wipe the 20 test points upserted in Step 4 before running the full production embed.

**Why:** The full embed script uses upsert, which overwrites existing points by ID — so stale test data won't cause errors. But starting from a clean slate is good practice.

**When to run:** Once only — right before kicking off the full embed in the terminal.

After running this cell, start the full embed with:
```
caffeinate -dimsu python scripts/sgg_text_embed.py embed
```

In [9]:
from qdrant_client import QdrantClient
from qdrant_client.http import models as rest

client = QdrantClient(url=QDRANT_URL)

client.delete_collection('sgg_text_v1')
print('Deleted sgg_text_v1 collection.')

client.create_collection(
    collection_name='sgg_text_v1',
    vectors_config=rest.VectorParams(size=768, distance=rest.Distance.COSINE),
)
for field in ('artist', 'genre', 'key_key', 'key_scale'):
    try:
        client.create_payload_index('sgg_text_v1', field, rest.PayloadSchemaType.KEYWORD)
    except Exception:
        pass

info = client.get_collection('sgg_text_v1')
print(f'Recreated sgg_text_v1 — points: {info.points_count}')
print('Ready for full embed.')


Deleted sgg_text_v1 collection.
Recreated sgg_text_v1 — points: 0
Ready for full embed.
